# B.4 任務完成後怎麼停止？


工具已算出2+3=5，系統還要繼續要求工具嗎？需要一個明確的完成信號；同時，也要設上限，避免一直重複請求。前置是[工具返回與執行軌跡](https://birdhackor.github.io/tiny-perceptron-vlm/B.3.html)。完成、被上限截停、請求錯誤與尚缺後續輸出，是四種不同結局，不能全部寫成「已回答」。

本地`tool_loop`接收預先列好的請求清單，用它測試外層流程，不包含模型。第一筆要求add，a=2、b=3；第二筆`done=True`宣告完成並提供answer=5。max_steps設定最多可處理幾筆，本工具也把完成訊息算作一筆。所以只允許一筆時，雖然加法已算完，第二筆完成訊息仍會被上限擋住。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.retrieval import tool_loop

requests = [
    {"name": "add", "arguments": {"a": 2, "b": 3}},
    {"done": True, "answer": 5},
]
runs = [
    tool_loop(requests, max_steps=3),
    tool_loop(requests, max_steps=1),
    tool_loop(requests[:1], max_steps=3),
    tool_loop([{"name": "unknown", "arguments": {"a": 2, "b": 3}}]),
]
for run in runs:
    print(run["status"], "工具執行筆數", len(run["trace"]), "答案", run.get("answer"))

輸出依序是done、step_limit、needs_more_model_output、invalid_request；工具執行筆數依序1、1、1、0，只有第一行答案是5，其餘是None。`requests[:1]`只留下第一筆：清單耗盡時仍沒有done，表示需要更多模型輸出，不會替模型自動宣告完成。最後一份未知工具在驗證階段被拒絕，沒有合法工具結果可加入trace。

done是這份協議的完成宣告，不是答案正確的保證；如果宣告answer=6，這個小循環仍會回傳done。正式系統還需用任務判準核對，例如本題真值是5。step_limit表示流程被預算截停，不表示題目一定無解；invalid_request則保存錯誤原因，可能需要修正名稱或參數後再試。三者的處理方式不同，保存status可以避免把截停當成功或把格式錯誤藏起來。

真實模型版本不是一開始就有整份requests，而是每次生成一個新請求，驗證執行後把結果送回，再生成下一個動作。這時要統計成功率、真正工具執行次數與等待時間，並保留每次輸入輸出。工具只做出被請求的運算，循環也只管理順序；這兩部分都不能代替模型理解何時已解決原問題。

練習只把第二次run的max_steps由1改成2。先預測它現在能讀到完成訊息，status變done、答案變5，工具執行筆數仍是1，再執行核對。這說明放寬的是可處理訊息數，不是強迫多算一次，也不是提高答案的數學正確率。
